In [2]:
%pip install casadi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 MB 7.3 MB/s  0:00:05a 0:00:010:00:01:01
Note: you may need to restart the kernel to use updated packages.


# 1. Etude du problème d’optimisation #
**Question 1 :** Justifier la forme de la dynamique de charge (1).

**Réponse :**
$\newline$


L'état de charge de la batterie à l'instant $i+1$ pour le véhicule $j$ correspond à l'état de charge à l'instant $i$ (i.e. $Q_{j,i}$) plus la charge apportée durant $∆t$ (entre $t_{i}$ et $t_{i+1}$). 
$\newline$
Cette charge apportée correspond à  $∆tβ_{j}P_{j,i}$ où : $∆t$ correspond au temps de charge, $P_{j,i}$ correspond à la puissance de la station de chargement $j$ et $β_{j}$ correspond au rendement de la batterie/au pourcentage de conversion de l'énergie de la zone de charge vers la batterie. 

**Question 2 :** Formuler le problème d’optimisation à résoudre sous la forme (5) :

$min_{x}$  $f(x)$  tel que :  $\quad c_{eq}(x) = 0\quad         c_{in}(x) ≤ 0$             

On précisera les variables de décision $x$, leur nombre $n$, les contraintes $c_{eq}$ et $c_{in}$ ainsi que la fonction
objectif $f$ à minimiser.*

**Réponse :**
$\newline$


On prend $P_{i,j}, Q_{i,j}$ pour $i\in[0, …, N-1]$ et $j\in[1, …, N_{v}]$, comme variables de décisions. 
$\newline$
On a donc $2.N.N_{v}$ vraiables.
$\newline$
La quantitée qu'on cherche à minimiser est la suivante : $$∆t\sum\limits_{j=1}^{N_{v}} \sum\limits_{i=0}^N c_{i}P_{i,j}\newline$$
On peut donc poser : $$f~:~x\longrightarrow∆t\sum\limits_{j=1}^{N_{v}} \sum\limits_{i=0}^N c_{i}x$$
$\newline$
Concernant les contraintes : 
- $\forall~j, ~ Q_{j, i_{jf}} - \bar{Q} = 0$
- $\forall~i, ~ \sum\limits_{j=1}^{N_{v}} P_{i,j} - \bar{P} \leq 0$
- $\forall~j,~\forall~i,~ Q_{j,i}-\bar{Q_{j}} \leq 0$
- $\forall~j,~\forall~i,~ P_{j,i} \geq 0$

/!\ j'ai des contraintes en trop, surement les dernières car on a des degrés de libertés négatifs....
Je suis pas sur pour f(x) aussi

**Question 3 :** Etudier la convexité de ce problème. Appartient-il à une famille particulière de problèmes d’optimisation ?

**Réponse :**
$\newline$



# 2. Identification du paramètre β #
**Question 4 :** On cherche à identifier le paramètre β à partir de ces essais, connaissant la charge maximale atteignable
$\bar{Q}$ (qui est elle simple à déterminer) et la tension secteur $U_{sec} = 230V$ , reliant puissance et intensité.
Formuler le problème de moindres carrés correspondant à l’identification de β.

**Question 5 :** Charger les données à l’aide du fichier data_battery.csv et résoudre ce problème pour $\bar{Q} = 50Ah$.
On pourra utiliser la solution des moindres carrés, ou utiliser la fonction numpy.linalg.lstsq. On
n’oubliera pas de filtrer convenablement le bruit de mesure, si besoin, et de commenter l’impact de ce
filtrage sur les résultats obtenus. On tracera en particulier sur la même figure l’évolution de l’état de
charge de la batterie au cours du temps, ainsi que l’état de charge estimé obtenu à partir de l’intensité
et du paramètre estimé β.

In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import casadi as ca

df = pd.read_csv('data_battery.csv')
df
# erreur qu'à relever Robin me semble t-il (demander à pauline et ines) mais le csv est à l'envers : les batteries se décharqgent.

,Time [s],SOC [%],I [kA]
0,0.000000,0.631635,0.000153
1,1.005016,0.631646,0.000951
2,2.008215,0.630583,-0.002994
3,3.018137,0.630439,0.000654
4,4.023168,0.632332,0.002940
...,...,...,...
294,295.401837,0.262350,-0.482445
295,296.407886,0.259741,-0.482207
296,297.413096,0.258567,-0.484523
297,298.418893,0.258645,-0.480281


# 3. Etude et résolution numérique du problème centralisé #
**Question 6 :** Quelles méthodes de résolution peuvent être envisagées pour ce problème ? 

**Question 7 :** Développer un algorithme de résolution dans le cas d’un horizon de temps de $24h (t_{0} = 17h, ∆t =
0.25h)$, avec :
$\newline$
• un tarif d’heure creuse ($c_{i} = c_{cr}$ constant) entre d’une part minuit et $6h$, et d’autre part $12h$ et $14h$
et un tarif d’heure pleine ($c_{i} = c_{pl}$ constant) le reste du temps ;
$\newline$
• un premier véhicule qui arrive avec un état de charge de $30%$ à $18h$ et souhaite repartir à $8h$ le
lendemain matin ;
$\newline$
• un second véhicule qui arrive avec un état de charge de $50%$ à $18h30$ et souhaite repartir à $9h$ le
lendemain.
$\newline$
On prendra les valeurs numériques suivantes :
$\newline$
$$c_{cr} = 1 , c_{pl} = 3/2 , \bar{Q} = 50Ah , \bar{P} = 2.5kW (6)$$
$\newline$
On veillera à afficher au minimum les graphiques suivants :$\newline$
(a) l’évolution de l’état de charge de chaque véhicule au cours du temps ainsi que la puissance (normalisée) fournie à chaque véhicule. On prendra soin d’indiquer les temps de contrainte de pleine
charge des véhicules.
$\newline$
(b) l’évolution de la puissance totale (normalisée) fournie à la flotte, au cours du temps, ainsi que
celle du tarif électrique.
$\newline$
Commenter les résultats obtenus.

**Question 8:** Quels inconvénients présente cette solution centralisée ? (On pourra penser aux données dont doit
disposer l’optimiseur pour la résolution.)